In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import numpy as np
import json, requests
from datetime import datetime
from kaggle_secrets import UserSecretsClient

In [11]:

SHEET_URL = "https://script.google.com/macros/s/AKfycbyjpRwDy8bUdOGobfckfVuHy-32GNCxkmX1q-yxRiKKZqzsVmXn-123sYgMzVSw1ev1og/exec"

In [24]:
DATASET_PATH  = '/kaggle/input/datasets/madhvii0911/datasets2/processed_dataset_v3'
NUM_CLASSES   = 4
BATCH_SIZE    = 32
MAX_EPOCHS    = 40
PATIENCE      = 5
IMG_SIZE      = 224
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [5]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=f'{DATASET_PATH}/train', transform=transform)
val_dataset   = ImageFolder(root=f'{DATASET_PATH}/test',  transform=transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)


In [6]:
class VGG16_Custom(nn.Module):
    def __init__(self, num_classes=4, dropout=0.5):
        super(VGG16_Custom, self).__init__()
        vgg = models.vgg16(pretrained=True)
        for param in vgg.features[:24].parameters():
            param.requires_grad = False
        self.features   = vgg.features
        self.avgpool    = vgg.avgpool
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(dropout),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [19]:
def train_model(model, optimizer, criterion, exp_id, phase, lr,
                patience=PATIENCE, max_epochs=MAX_EPOCHS, notes=''):

    model = model.to(DEVICE)

    train_accs, val_accs     = [], []
    train_losses, val_losses = [], []
    best_val_acc = 0.0
    patience_counter = 0

    for epoch in range(max_epochs):
        # --- Train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
        train_accs.append(correct / total)
        train_losses.append(running_loss / len(train_loader))

        # --- Validate ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss    += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total   += labels.size(0)
        val_accs.append(val_correct / val_total)
        val_losses.append(val_loss / len(val_loader))

        print(f"Epoch {epoch+1}/{max_epochs} | "
              f"Train Acc: {train_accs[-1]*100:.2f}% | "
              f"Val Acc: {val_accs[-1]*100:.2f}% | "
              f"Train Loss: {train_losses[-1]:.4f} | "
              f"Val Loss: {val_losses[-1]:.4f}")

        # --- Early Stopping ---
        if val_accs[-1] > best_val_acc:
            best_val_acc     = val_accs[-1]
            patience_counter = 0
            torch.save(model.state_dict(), f'{exp_id}_best.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"⏹ Early stopped at epoch {epoch+1}")
                break
    log_to_sheet(
        exp_id       = exp_id,
        dataset      = 'preprocessed_v3',
        
        model        = model,
        preprocessing = 'Resize 224 * 224,Normalize(ImageNet mean/std),320 train / 80 test',
        train_accs   = train_accs,
        val_accs     = val_accs,
        train_losses = train_losses,
        val_losses   = val_losses,
        phase        = phase,
        lr           = lr,
        patience     = patience,
        max_epochs   = max_epochs,
        batch_size   = BATCH_SIZE,
        img_size     = f'{IMG_SIZE}x{IMG_SIZE}',
        notes        = notes
    )

    return model, train_accs, val_accs, train_losses, val_losses

# ============================================================
# LOG TO SHEET
# ============================================================
def log_to_sheet(exp_id, dataset, preprocessing, model,
                 train_accs, val_accs, train_losses, val_losses,
                 phase, lr, patience, max_epochs,
                 batch_size, img_size, notes=''):

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params    = total_params - trainable_params
    total_layers     = len(list(model.modules()))
    trainable_layers = len([l for l in model.modules()
                            if any(p.requires_grad for p in l.parameters(recurse=False))])
    frozen_layers    = total_layers - trainable_layers

    best_epoch    = int(np.argmax(val_accs)) + 1
    epochs_run    = len(val_accs)
    early_stopped = epochs_run < max_epochs

    row = [
        exp_id,
        datetime.now().strftime('%Y-%m-%d %H:%M'),
        dataset, preprocessing,
        model.__class__.__name__,
        f"{total_params:,}", f"{trainable_params:,}", f"{frozen_params:,}",
        frozen_layers, trainable_layers,
        img_size, batch_size, phase, str(lr), patience,
        epochs_run, max_epochs,
        f"{max(train_accs)*100:.2f}%",
        f"{max(val_accs)*100:.2f}%",
        f"{min(train_losses):.4f}",
        f"{min(val_losses):.4f}",
        best_epoch, str(early_stopped), notes
    ]

    response = requests.post(SHEET_URL,
                             data=json.dumps({"row": row}),
                             headers={"Content-Type": "application/json"})
    print(f"✅ Logged: {exp_id} | Val Acc: {max(val_accs)*100:.2f}% | Best Epoch: {best_epoch} | Early Stop: {early_stopped}")
    print(f"   Response: {response.text}")

# ============================================================
# RUN EXPERIMENTS
# ============================================================
criterion = nn.CrossEntropyLoss()

experiments = [
    ('EXP_001', 0.001, 'adam',  'Transfer Learning - Adam lr=0.001'),
    ('EXP_002', 0.0001,'adam',  'Transfer Learning - Adam lr=0.0001'),
    ('EXP_003', 0.01,  'sgd',   'Transfer Learning - SGD lr=0.01'),
]

for exp_id, lr, opt_type, notes in experiments:
    print(f"\n{'='*50}")
    print(f"Running {exp_id} | {notes}")
    print(f"{'='*50}")

    model = VGG16_Custom(num_classes=NUM_CLASSES).to(DEVICE)

    if opt_type == 'adam':
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, momentum=0.9)

    train_model(model, optimizer, criterion,
                exp_id=exp_id, phase='Transfer Learning',
                lr=lr, notes=notes)



Running EXP_001 | Transfer Learning - Adam lr=0.001
Epoch 1/20 | Train Acc: 25.62% | Val Acc: 25.00% | Train Loss: 1.5372 | Val Loss: 1.3865
Epoch 2/20 | Train Acc: 25.47% | Val Acc: 25.00% | Train Loss: 1.3884 | Val Loss: 1.3866
Epoch 3/20 | Train Acc: 24.61% | Val Acc: 25.00% | Train Loss: 1.3871 | Val Loss: 1.3863
Epoch 4/20 | Train Acc: 24.53% | Val Acc: 25.00% | Train Loss: 1.3874 | Val Loss: 1.3863
Epoch 5/20 | Train Acc: 25.00% | Val Acc: 25.00% | Train Loss: 1.3865 | Val Loss: 1.3864
Epoch 6/20 | Train Acc: 24.22% | Val Acc: 25.00% | Train Loss: 1.3867 | Val Loss: 1.3863
⏹ Early stopped at epoch 6
✅ Logged: EXP_001 | Val Acc: 25.00% | Best Epoch: 1 | Early Stop: True
   Response: OK

Running EXP_002 | Transfer Learning - Adam lr=0.0001
Epoch 1/20 | Train Acc: 51.56% | Val Acc: 46.88% | Train Loss: 1.0597 | Val Loss: 1.3323
Epoch 2/20 | Train Acc: 79.77% | Val Acc: 48.44% | Train Loss: 0.4775 | Val Loss: 2.2579
Epoch 3/20 | Train Acc: 94.30% | Val Acc: 54.37% | Train Loss: 0.16

In [20]:
# ============================================================
# AUGMENTED TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, 
                           saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Val/Test pe augmentation nahi — sirf resize + normalize
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=f'{DATASET_PATH}/train', transform=train_transform)
val_dataset   = ImageFolder(root=f'{DATASET_PATH}/test',  transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# EXPERIMENTS WITH AUGMENTATION
# ============================================================
experiments = [
   
    ('EXP_005', 0.0001, 'adam', 'Augmented - Adam lr=0.0001'),
    ('EXP_006', 0.01,   'sgd',  'Augmented - SGD lr=0.01'),
]

for exp_id, lr, opt_type, notes in experiments:
    print(f"\n{'='*50}")
    print(f"Running {exp_id} | {notes}")
    print(f"{'='*50}")

    model = VGG16_Custom(num_classes=NUM_CLASSES).to(DEVICE)

    if opt_type == 'adam':
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, momentum=0.9)

    train_model(model, optimizer, criterion,
                exp_id=exp_id, phase='Augmented Transfer Learning',
                lr=lr, notes=notes)


Running EXP_005 | Augmented - Adam lr=0.0001
Epoch 1/20 | Train Acc: 42.19% | Val Acc: 41.25% | Train Loss: 1.2037 | Val Loss: 1.5327
Epoch 2/20 | Train Acc: 52.73% | Val Acc: 44.38% | Train Loss: 1.0007 | Val Loss: 1.5340
Epoch 3/20 | Train Acc: 61.88% | Val Acc: 41.25% | Train Loss: 0.8309 | Val Loss: 1.8959
Epoch 4/20 | Train Acc: 65.16% | Val Acc: 44.06% | Train Loss: 0.7682 | Val Loss: 1.6151
Epoch 5/20 | Train Acc: 69.06% | Val Acc: 50.94% | Train Loss: 0.7018 | Val Loss: 1.5423
Epoch 6/20 | Train Acc: 70.08% | Val Acc: 47.81% | Train Loss: 0.6695 | Val Loss: 1.5760
Epoch 7/20 | Train Acc: 72.03% | Val Acc: 50.31% | Train Loss: 0.6060 | Val Loss: 1.7803
Epoch 8/20 | Train Acc: 73.83% | Val Acc: 54.37% | Train Loss: 0.5921 | Val Loss: 1.4813
Epoch 9/20 | Train Acc: 74.30% | Val Acc: 49.38% | Train Loss: 0.5678 | Val Loss: 2.0546
Epoch 10/20 | Train Acc: 77.11% | Val Acc: 50.31% | Train Loss: 0.5335 | Val Loss: 1.7709
Epoch 11/20 | Train Acc: 77.19% | Val Acc: 58.13% | Train Loss:

In [21]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees = 15),
    transforms.ColorJitter(brightness=0.3,contrast=0.3,
                          saturation = 0.2,hue = 0.1),
    transforms.RandomAffine(degrees = 0,translate = (0.1,0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                        std = [0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=f'{DATASET_PATH}/train', transform=train_transform)
val_dataset   = ImageFolder(root=f'{DATASET_PATH}/test',  transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)




In [25]:
class VGG16_Custom(nn.Module):
    def __init__(self, num_classes=4, dropout=0.5):
        super(VGG16_Custom, self).__init__()
        vgg = models.vgg16(pretrained=True)
        for param in vgg.features[:24].parameters():
            param.requires_grad = False
        self.features   = vgg.features
        self.avgpool    = vgg.avgpool
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(dropout),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# ============================================================
# EXPERIMENTS
# ============================================================

criterion = nn.CrossEntropyLoss()

experiments = [
    # (exp_id, lr, optimizer, dropout, weight_decay, notes)
    ('EXP_007', 0.01,  'sgd', 0.5, 0,    'SGD + Augmentation'),
    ('EXP_008', 0.01,  'sgd', 0.5, 1e-4, 'SGD + Augmentation + Weight Decay'),
    ('EXP_009', 0.01,  'sgd', 0.7, 0,    'SGD + Augmentation + Dropout 0.7'),
    ('EXP_010', 0.01,  'sgd', 0.7, 1e-4, 'SGD + Aug + Dropout 0.7 + Weight Decay'),
]

for exp_id, lr, opt_type, dropout, weight_decay, notes in experiments:
    print(f"\n{'='*50}")
    print(f"Running {exp_id} | {notes}")
    print(f"{'='*50}")

    model = VGG16_Custom(num_classes=NUM_CLASSES, dropout=dropout).to(DEVICE)

    if opt_type == 'adam':
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, momentum=0.9, weight_decay=weight_decay)

    train_model(model, optimizer, criterion,
                exp_id=exp_id, phase='Augmented + Regularization',
                lr=lr, notes=notes)


Running EXP_007 | SGD + Augmentation
Epoch 1/20 | Train Acc: 37.66% | Val Acc: 33.44% | Train Loss: 1.3045 | Val Loss: 1.5414
Epoch 2/20 | Train Acc: 46.25% | Val Acc: 36.88% | Train Loss: 1.1540 | Val Loss: 1.3557
Epoch 3/20 | Train Acc: 51.80% | Val Acc: 39.69% | Train Loss: 1.0375 | Val Loss: 1.3660
Epoch 4/20 | Train Acc: 56.41% | Val Acc: 41.56% | Train Loss: 0.9552 | Val Loss: 1.5167
Epoch 5/20 | Train Acc: 54.53% | Val Acc: 42.19% | Train Loss: 0.9501 | Val Loss: 1.4381
Epoch 6/20 | Train Acc: 60.47% | Val Acc: 42.19% | Train Loss: 0.8397 | Val Loss: 1.8969
Epoch 7/20 | Train Acc: 64.22% | Val Acc: 45.94% | Train Loss: 0.8173 | Val Loss: 1.4713
Epoch 8/20 | Train Acc: 64.06% | Val Acc: 51.56% | Train Loss: 0.8061 | Val Loss: 1.2599
Epoch 9/20 | Train Acc: 65.08% | Val Acc: 44.69% | Train Loss: 0.7597 | Val Loss: 1.3004
Epoch 10/20 | Train Acc: 69.14% | Val Acc: 52.81% | Train Loss: 0.7033 | Val Loss: 1.4935
Epoch 11/20 | Train Acc: 68.98% | Val Acc: 46.56% | Train Loss: 0.6942 

In [17]:
def log_to_sheet(exp_id, dataset, preprocessing, model,
                 train_accs, val_accs, train_losses, val_losses,
                 phase, lr, patience, max_epochs,
                 batch_size, img_size, notes=''):

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params    = total_params - trainable_params
    total_layers     = len(list(model.modules()))
    trainable_layers = len([l for l in model.modules()
                            if any(p.requires_grad for p in l.parameters(recurse=False))])
    frozen_layers    = total_layers - trainable_layers

    best_epoch    = int(np.argmax(val_accs)) + 1
    epochs_run    = len(val_accs)
    early_stopped = epochs_run < max_epochs

    row = [
        exp_id,
        datetime.now().strftime('%Y-%m-%d %H:%M'),
        dataset, preprocessing,
        model.__class__.__name__,
        f"{total_params:,}", f"{trainable_params:,}", f"{frozen_params:,}",
        frozen_layers, trainable_layers,
        img_size, batch_size, phase, str(lr), patience,
        epochs_run, max_epochs,
        f"{max(train_accs)*100:.2f}%",
        f"{max(val_accs)*100:.2f}%",
        f"{min(train_losses):.4f}",
        f"{min(val_losses):.4f}",
        best_epoch, str(early_stopped), notes
    ]

    response = requests.post(SHEET_URL,
                             data=json.dumps({"row": row}),
                             headers={"Content-Type": "application/json"})
    print(f"✅ Logged: {exp_id} | Val Acc: {max(val_accs)*100:.2f}% | Best Epoch: {best_epoch} | Early Stop: {early_stopped}")
    print(f"   Response: {response.text}")
